# Module 4: DataFrame Transformations

This is the **core module**. These 7 operations let you do real data analysis:

| Operation | What it does | SQL equivalent |
|-----------|-------------|----------------|
| `select` | Pick/compute columns | `SELECT` |
| `filter`/`where` | Pick rows | `WHERE` |
| `withColumn` | Add/modify columns | `SELECT *, expr AS new_col` |
| `groupBy` + `agg` | Aggregate | `GROUP BY ... AGG()` |
| `orderBy` | Sort | `ORDER BY` |
| `join` | Combine tables | `JOIN` |

Master these and you can answer any question about your data.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, upper, round, count, sum, avg, min, max

spark = SparkSession.builder \
    .appName("Module 04 - DataFrame Transforms") \
    .master("local[*]") \
    .getOrCreate()

employees = spark.read.csv("../data/employees.csv", header=True, inferSchema=True)
departments = spark.read.csv("../data/departments.csv", header=True, inferSchema=True)
sales = spark.read.csv("../data/sales.csv", header=True, inferSchema=True)

---
## Concept 1: select — Picking and Computing Columns

Like SQL's `SELECT`. You can pick existing columns, rename them with `.alias()`, or compute new ones with expressions.

`col("name")` creates a column reference. You can do arithmetic, string operations, and more on column references.

In [ ]:
# Simple column selection
employees.select("name", "salary").show(5)

In [ ]:
# Computed columns with expressions and aliases
employees.select(
    col("name"),
    col("salary"),
    (col("salary") * 1.1).alias("salary_with_raise"),
    upper(col("name")).alias("name_upper")
).show(5)

#### Try It: Select with Computed Columns

1. Select `name`, `salary`, and a new column `salary_with_15pct_raise` (salary * 1.15)
2. Use `.alias()` to name it properly
3. Show the first 5 rows

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
employees.select(
    col("name"),
    col("salary"),
    (col("salary") * 1.15).alias("salary_with_15pct_raise")
).show(5)

---
## Concept 2: filter / where — Selecting Rows

`filter()` and `where()` are identical — use whichever reads better to you.

**Important syntax**: Use `&` for AND, `|` for OR, `~` for NOT. **Always wrap each condition in parentheses** — Python operator precedence requires this.

In [ ]:
# Single condition
high_earners = employees.filter(col("salary") > 90000)
print(f"Employees earning > $90k: {high_earners.count()}")
high_earners.select("name", "salary", "city").show()

In [ ]:
# Multiple conditions — parentheses are REQUIRED around each condition
sf_high = employees.filter(
    (col("salary") > 90000) & (col("city") == "San Francisco")
)
sf_high.select("name", "salary", "city").show()

#### Try It: Filter Rows

Find all employees in **Chicago** earning **more than $65,000**. Show their name, salary, and city.

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
employees.filter(
    (col("city") == "Chicago") & (col("salary") > 65000)
).select("name", "salary", "city").show()

---
## Concept 3: withColumn — Adding or Modifying Columns

`withColumn(name, expression)` adds a new column or replaces an existing one. Combined with `when().otherwise()`, you get conditional logic like SQL's `CASE WHEN`.

In [ ]:
# Conditional column with when/otherwise
employees_banded = employees.withColumn(
    "salary_band",
    when(col("salary") >= 100000, "Senior")
    .when(col("salary") >= 80000, "Mid")
    .otherwise("Junior")
).withColumn(
    "bonus", round(col("salary") * 0.1, 2)
)

employees_banded.select("name", "salary", "salary_band", "bonus").show(10)

#### Try It: Add Conditional Columns

1. Add a `tax` column that is 30% of salary
2. Add a `seniority` column: `"Veteran"` if hired before 2019, `"Regular"` if 2019-2020, `"New"` if after 2020

Hint: `hire_date` is a date column, so use `col("hire_date") < "2019-01-01"`

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
result = employees \
    .withColumn("tax", round(col("salary") * 0.30, 2)) \
    .withColumn("seniority",
        when(col("hire_date") < "2019-01-01", "Veteran")
        .when(col("hire_date") <= "2020-12-31", "Regular")
        .otherwise("New")
    )
result.select("name", "hire_date", "salary", "tax", "seniority").show(10)

---
## Concept 4: groupBy + agg — Aggregations

Group rows by one or more columns, then compute aggregate statistics. This is how you answer questions like "What is the average salary per city?" or "Total revenue per product?"

Common aggregate functions: `count()`, `sum()`, `avg()`, `min()`, `max()`

In [ ]:
# Employees per city with salary stats
employees.groupBy("city").agg(
    count("*").alias("num_employees"),
    round(avg("salary"), 2).alias("avg_salary"),
    min("salary").alias("min_salary"),
    max("salary").alias("max_salary")
).show()

In [ ]:
# Sales by product
sales.groupBy("product").agg(
    round(sum("amount"), 2).alias("total_revenue"),
    sum("quantity").alias("total_units"),
    count("*").alias("num_sales")
).orderBy(col("total_revenue").desc()).show()

#### Try It: Aggregate Data

Group employees by `department_id` and compute:
- Number of employees
- Average salary (rounded to 2 decimals)
- Total salary (sum)

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
employees.groupBy("department_id").agg(
    count("*").alias("num_employees"),
    round(avg("salary"), 2).alias("avg_salary"),
    sum("salary").alias("total_salary")
).orderBy("department_id").show()

---
## Concept 5: orderBy — Sorting Results

Sort by one or more columns. Use `.desc()` for descending order.

In [ ]:
# Top 5 highest-paid employees
employees.orderBy(col("salary").desc()).show(5)

In [ ]:
# Sort by multiple columns: city ascending, then salary descending within each city
employees.orderBy("city", col("salary").desc()).select("city", "name", "salary").show(10)

#### Try It: Sort Data

1. Find the **top 5 largest sales** by amount (descending)
2. Sort employees by `hire_date` ascending — who was hired first?

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
print("Top 5 sales:")
sales.orderBy(col("amount").desc()).show(5)

print("Earliest hires:")
employees.orderBy("hire_date").select("name", "hire_date", "salary").show(5)

---
## Concept 6: join — Combining DataFrames

Joins connect two DataFrames by matching column values. This is how you combine related tables — employees with their departments, sales with the salesperson's name, etc.

Join types:
- `inner` (default) — only rows that match in BOTH tables
- `left` — all rows from left + matching from right (nulls where no match)
- `right` — all rows from right + matching from left
- `outer` — all rows from both tables

**In Big Data, joins are expensive** — they often require a shuffle (moving data between machines). Always join on indexed/partitioned columns when possible.

In [ ]:
# Join employees with departments on department_id
emp_dept = employees.join(departments, on="department_id", how="inner")

emp_dept.select("name", "department_name", "salary", "location").show(10)

In [ ]:
# Multi-table join: sales -> employees -> departments
full_data = (
    sales
    .join(employees, on="employee_id")
    .join(departments, on="department_id")
)

full_data.select("name", "department_name", "product", "amount", "sale_date").show(10)

#### Try It: Join Tables

Join employees to departments and show: `name`, `department_name`, `salary`, `budget`

Then filter for employees whose salary is more than 5% of their department's budget.

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
emp_dept = employees.join(departments, on="department_id")

emp_dept.select("name", "department_name", "salary", "budget").show(5)

print("Employees whose salary > 5% of department budget:")
emp_dept.filter(
    col("salary") > col("budget") * 0.05
).select("name", "department_name", "salary", "budget").show()

---
## Concept 7: Chaining It All Together

The real power comes from **chaining** operations. A single pipeline can join, filter, group, aggregate, and sort — answering complex business questions in one expression.

In [ ]:
# Which department generated the most revenue?
revenue_by_dept = (
    sales
    .join(employees, on="employee_id")
    .join(departments, on="department_id")
    .groupBy("department_name")
    .agg(
        round(sum("amount"), 2).alias("total_revenue"),
        count("*").alias("num_sales"),
        round(avg("amount"), 2).alias("avg_sale")
    )
    .orderBy(col("total_revenue").desc())
)

revenue_by_dept.show()

#### Try It: Revenue by Region

Build a pipeline that:
1. Joins sales to employees (on `employee_id`)
2. Groups by `region`
3. Computes `total_revenue` (sum of amount) and `num_sales` (count)
4. Orders by total_revenue descending

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
(
    sales
    .join(employees, on="employee_id")
    .groupBy("region")
    .agg(
        round(sum("amount"), 2).alias("total_revenue"),
        count("*").alias("num_sales")
    )
    .orderBy(col("total_revenue").desc())
    .show()
)

---
## Capstone Exercise

Find the **top 3 employees by total sales amount**, including their department name.

This requires:
- 3-table join (sales -> employees -> departments)
- groupBy employee name (and department_name)
- agg: sum of amount, count of sales
- orderBy total descending
- show top 3

In [ ]:
# Your capstone code here


In [ ]:
# --- Capstone Solution ---
top_sellers = (
    sales
    .join(employees, on="employee_id")
    .join(departments, on="department_id")
    .groupBy("name", "department_name")
    .agg(
        round(sum("amount"), 2).alias("total_sales"),
        count("*").alias("num_transactions")
    )
    .orderBy(col("total_sales").desc())
)

print("Top 3 sellers:")
top_sellers.show(3)

In [ ]:
spark.stop()

---
## What You Learned

- **`select`** picks/computes columns, **`filter`** picks rows
- **`withColumn`** + **`when/otherwise`** for conditional logic
- **`groupBy` + `agg`** for aggregations (count, sum, avg, min, max)
- **`orderBy`** for sorting (`.desc()` for descending)
- **`join`** combines DataFrames (inner, left, right, outer)
- **Chaining** operations builds powerful analytical pipelines

Next: **Module 5 — Spark SQL**, where you'll write the same queries in SQL.